In [1]:
# =========================================================
# Conversational AI Project (T5 + DailyDialog)
# =========================================================

# =========================
# 1. IMPORTS
# =========================

import torch
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration
)

# =========================
# 2. LOAD DATASET
# =========================

dataset = load_dataset("blended_skill_talk")

print(dataset["train"][0])

# =========================
# 3. BUILD CONVERSATION PAIRS
# =========================

pairs = []

for item in dataset["train"]:

    messages = item["previous_utterance"] + item["free_messages"]

    for i in range(len(messages) - 1):

        question = messages[i]
        answer = messages[i + 1]

        pairs.append((question, answer))


pairs = pairs[:2000]

print("Total pairs:", len(pairs))

print("\nExamples:\n")
for i in range(5):
    print("Q:", pairs[i][0])
    print("A:", pairs[i][1])
    print()

# =========================
# 4. LOAD TOKENIZER
# =========================

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")

# =========================
# 5. PREPROCESS
# =========================

MAX_LEN = 64

def preprocess(question, answer):

    input_text = "dialogue: " + question
    target_text = answer

    inputs = tokenizer(
        input_text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    targets = tokenizer(
        target_text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"].squeeze(),
        "attention_mask": inputs["attention_mask"].squeeze(),
        "labels": targets["input_ids"].squeeze()
    }

# =========================
# 6. DATASET CLASS
# =========================

class ChatDataset(Dataset):

    def __init__(self, pairs):
        self.data = pairs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        question, answer = self.data[idx]

        return preprocess(question, answer)

# =========================
# 7. DATALOADER
# =========================

train_dataset = ChatDataset(pairs)

loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

# =========================
# 8. LOAD MODEL
# =========================

model = T5ForConditionalGeneration.from_pretrained(
    "google/flan-t5-small"
)

# =========================
# 9. OPTIMIZER
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5
)

# =========================
# 10. TRAINING
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

epochs = 1

print("\nTraining Started...\n")

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# =========================
# 11. CHAT FUNCTION
# =========================

def chat(text):

    model.eval()

    input_text = "dialogue: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_length=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

# =========================
# 12. TEST
# =========================

print("\n====================")
print("CHAT TEST")
print("====================\n")

print("User: Hello")
print("Bot :", chat("Hello"))

print()

print("User: How are you?")
print("Bot :", chat("How are you?"))

print()

print("User: What is your name?")
print("Bot :", chat("What is your name?"))

D:\Anaconda3\envs\chatbot_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'personas': ["i've 2 kids.", 'i love flowers.'], 'additional_context': '', 'previous_utterance': ["I love live music, that's why I try to go to concerts", 'I do too. Wat do you like?'], 'context': 'empathetic_dialogues', 'free_messages': ['I like acting, I hope to be an actor, what about you?', 'No, but someday.', 'After I am done with school I plan to have a family.', 'I hope so, how old are your kids?', 'I would imagine. I am sure they a great kids.', 'I wish I had more time to do stuff like that. Medical school is exhausting. '], 'guided_messages': ['that is ok.  have any kids?', 'that is good. I have 2', 'that is great! you will be ready', '5 & 7.  they take up a lot of my time', 'luckily, they love flowers just as much as I do.  we spend a lot of time in the garden', 'sounds like it. have you gotten any acting jobs, though?'], 'suggestions': {'convai2': ["i love acting ! i'll be famous someday . what do you do ?", 'no no kids , might get some though . one day', 'that is great . i

Loading weights: 100%|██████████████████████████████████████████████████████████████| 190/190 [00:00<00:00, 786.89it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Training Started...

Epoch 1, Loss: 2745.5781

CHAT TEST

User: Hello
Bot : In the upcoming season

User: How are you?
Bot : How do you know?

User: What is your name?
Bot : Bryse
